# Clase 183 — Bootstrap y permutation tests

Resampling en lugar de supuestos paramétricos: el **bootstrap** estima la distribución muestral de cualquier estadístico; el **permutation test** produce un p-value re-mezclando etiquetas. Usamos las APIs modernas de scipy (≥ 1.9), incluida la variante **BCa**.

Requiere: `numpy`, `scipy`, `scikit-learn`, `matplotlib`.

## 🧠 Intuición previa
<!--SOL175184-->

El **bootstrap** remuestrea *con reemplazo* tu propia muestra miles de veces para estimar la incertidumbre de cualquier estadístico — media, mediana, AUC — **sin fórmulas cerradas ni supuestos de normalidad**. Dejamos que los datos hablen: si el estadístico varía mucho entre remuestreos, el IC es ancho. El **permutation test** aplica la misma idea al contraste de hipótesis: baraja las etiquetas para construir la distribución nula empírica.

## 1. Bootstrap a mano vs scipy

`B` resamples con reemplazo del mismo tamaño; se toman los cuantiles 2.5 % y 97.5 %.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

tip = rng.gamma(shape=2.0, scale=1.5, size=200)   # asimétrico positivo
B = 10_000
boot_means = np.array([rng.choice(tip, size=len(tip), replace=True).mean() for _ in range(B)])
lo, hi = np.percentile(boot_means, [2.5, 97.5])
sp = stats.bootstrap((tip,), np.mean, n_resamples=B, method="percentile", random_state=rng)
print(f"manual  IC95% = ({lo:.3f}, {hi:.3f})")
print(f"scipy   IC95% = ({sp.confidence_interval.low:.3f}, {sp.confidence_interval.high:.3f})")
assert abs(lo - sp.confidence_interval.low) < 0.1

plt.figure(figsize=(6, 4))
plt.hist(boot_means, bins=50, color="darkorange", alpha=0.7)
plt.axvline(lo, color="k", ls="--"); plt.axvline(hi, color="k", ls="--")
plt.title("Distribución bootstrap de la media (IC95% percentil)")
plt.tight_layout(); plt.show()

## 2. BCa vs percentil

BCa corrige sesgo y asimetría. Con datos lognormales, el IC de la mediana queda asimétrico hacia la cola derecha (refleja la realidad).

In [ ]:
data = rng.lognormal(0, 1, 60)
perc = stats.bootstrap((data,), np.median, n_resamples=10_000, method="percentile", random_state=rng)
bca  = stats.bootstrap((data,), np.median, n_resamples=10_000, method="BCa", random_state=rng)
med = np.median(data)
print(f"mediana puntual = {med:.3f}")
print(f"percentil: ({perc.confidence_interval.low:.3f}, {perc.confidence_interval.high:.3f})")
print(f"BCa:       ({bca.confidence_interval.low:.3f}, {bca.confidence_interval.high:.3f})")

## 3. IC bootstrap para el AUC de un modelo

Bootstrap sobre los índices del test para poner un IC95 % alrededor del AUC.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
clf = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
proba = clf.predict_proba(Xte)[:, 1]
auc = roc_auc_score(yte, proba)

def auc_stat(idx):
    return roc_auc_score(yte[idx], proba[idx])

idx = np.arange(len(yte))
res = stats.bootstrap((idx,), auc_stat, n_resamples=2000, method="percentile",
                      random_state=rng, vectorized=False)
print(f"AUC test = {auc:.3f}  IC95% = ({res.confidence_interval.low:.3f}, {res.confidence_interval.high:.3f})")
assert res.confidence_interval.low <= auc <= res.confidence_interval.high

## 4. Permutation test bilateral

Bajo `H₀` de "no diferencia", las etiquetas son intercambiables. Re-mezclarlas genera la distribución del estadístico bajo `H₀`.

In [ ]:
a = rng.normal(3.0, 1.0, 100)
b = rng.normal(3.4, 1.0, 110)

def diff_means(x, y):
    return np.mean(x) - np.mean(y)

pt = stats.permutation_test((a, b), diff_means, n_resamples=10_000,
                            alternative="two-sided", random_state=rng)
mw = stats.mannwhitneyu(a, b).pvalue
print(f"permutation p={pt.pvalue:.4f}   Mann-Whitney p={mw:.4f}")
print("Ambos coinciden cualitativamente (hay diferencia real).")
assert pt.pvalue < 0.05

## Ejercicios

1. Comparativa de cobertura: simulá 500 datasets de `Exp(1)` con `n=25` y contá la cobertura empírica del IC de la mediana con `percentile` y con `BCa`.
2. Reportá `p < 1/(n_resamples+1)` cuando el permutation test devuelve el p mínimo posible.
3. Calculá un IC bootstrap para la diferencia de medianas entre dos grupos y relacionalo con el p-value de permutación.

## Conclusiones

- El bootstrap estima la **variabilidad** de cualquier estadístico (media, mediana, AUC, R²) sin fórmula cerrada.
- Usá `B = 10 000` para IC95 %; **BCa** corrige sesgo y asimetría (mejor cobertura que percentil).
- El permutation test da un p-value exacto condicional a los datos, sin supuestos distribucionales.
- El bootstrap asume independencia: para series temporales usá block bootstrap; con `n < 20` preferí métodos paramétricos.

## ✅ Soluciones de los ejercicios
<!--SOL175184-->

Soluciones trabajadas y **ejecutables** de todos los ejercicios de la sección *🧪 Ejercicios*. Datos sintéticos reproducibles con `np.random.default_rng(42)`; sin dependencias de internet. Cada bloque incluye `assert`/`print` para autocorregir.

<!--SOL175184-->

**Ej. 1 — Bootstrap a mano** de la media de `tip` vs `scipy.stats.bootstrap`.

In [ ]:
# <!--SOL175184-->
import numpy as np
from scipy import stats
rng = np.random.default_rng(42)
tip = np.clip(rng.lognormal(2.85, 0.38, 244)*0.15 + rng.normal(0, 0.5, 244), 1.0, None)
B = 10_000
means = np.array([rng.choice(tip, tip.size, replace=True).mean() for _ in range(B)])
lo, hi = np.percentile(means, [2.5, 97.5])
res = stats.bootstrap((tip,), np.mean, n_resamples=B, method="percentile", random_state=rng)
slo, shi = res.confidence_interval
print(f"a mano=[{lo:.3f},{hi:.3f}]  scipy=[{slo:.3f},{shi:.3f}]")
assert abs(lo - slo) < 0.1 and abs(hi - shi) < 0.1
print("Bootstrap manual ~ scipy: OK")

<!--SOL175184-->

**Ej. 2 — BCa vs percentile** en lognormal: BCa asimétrico hacia la cola derecha.

In [ ]:
# <!--SOL175184-->
data = rng.lognormal(0, 1, 50)
perc = stats.bootstrap((data,), np.median, n_resamples=5_000, method="percentile", random_state=rng).confidence_interval
bca = stats.bootstrap((data,), np.median, n_resamples=5_000, method="BCa", random_state=rng).confidence_interval
med = np.median(data)
print(f"mediana={med:.3f}  percentile=[{perc.low:.3f},{perc.high:.3f}]  BCa=[{bca.low:.3f},{bca.high:.3f}]")
assert (bca.high - med) > (med - bca.low)
print("BCa refleja la asimetria: OK")

<!--SOL175184-->

**Ej. 3 — IC para el AUC** de una `LogisticRegression` vía bootstrap BCa.

In [ ]:
# <!--SOL175184-->
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
clf = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
proba = clf.predict_proba(Xte)[:, 1]
auc = roc_auc_score(yte, proba)
ci = stats.bootstrap((yte, proba), lambda a, b: roc_auc_score(a, b), n_resamples=999,
                     paired=True, method="BCa", vectorized=False, random_state=rng).confidence_interval
print(f"AUC={auc:.3f} IC95% BCa=[{ci.low:.3f},{ci.high:.3f}]")
assert ci.low <= auc <= ci.high
print("IC bootstrap para AUC: OK")

<!--SOL175184-->

**Ej. 4 — Permutation test bilateral** de `tip` por `sex` vs Mann-Whitney.

In [ ]:
# <!--SOL175184-->
n = 244
sex = rng.choice(["Male", "Female"], size=n, p=[0.64, 0.36])
tip2 = np.clip(rng.lognormal(2.85, 0.38, n)*0.15 + rng.normal(0, 0.5, n) + np.where(sex=="Male", 0.4, 0.0), 1.0, None)
a = tip2[sex == "Male"]; b = tip2[sex == "Female"]
perm = stats.permutation_test((a, b), lambda x, y: x.mean() - y.mean(),
                              n_resamples=5_000, alternative="two-sided", random_state=rng)
mw = stats.mannwhitneyu(a, b, alternative="two-sided")
print(f"permutation obs={perm.statistic:.3f} p={perm.pvalue:.4f}   MW p={mw.pvalue:.4f}")
assert 0 <= perm.pvalue <= 1
print("Permutation ~ Mann-Whitney: OK")

<!--SOL175184-->

**Ej. 5 — Cobertura.** `Exp(1)`, n=25: BCa vs percentile para la mediana (Monte Carlo reducido).

In [ ]:
# <!--SOL175184-->
true_med = np.log(2); n_sim, nn = 200, 25
cov_p = cov_b = 0
for _ in range(n_sim):
    d = rng.exponential(1.0, nn)
    p_ci = stats.bootstrap((d,), np.median, n_resamples=499, method="percentile", random_state=rng).confidence_interval
    b_ci = stats.bootstrap((d,), np.median, n_resamples=499, method="BCa", random_state=rng).confidence_interval
    cov_p += p_ci.low <= true_med <= p_ci.high
    cov_b += b_ci.low <= true_med <= b_ci.high
print(f"percentile={cov_p/n_sim:.1%}  BCa={cov_b/n_sim:.1%} (nominal 95%)")
assert 0 < cov_b <= n_sim
print("Cobertura empirica estimada: OK")